<a href="https://colab.research.google.com/github/gvessio/pensare-in-python/blob/main/notebook/capitolo_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capitolo 7 — Iterazione e ricerca

Traduzione italiana del capitolo 7 di [*Think Python*, terza edizione](https://allendowney.github.io/ThinkPython/chap07.html), di **Allen B. Downey** (© 2024). Testo distribuito con licenza [Creative Commons BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/); codice con licenza [MIT](https://mit-license.org/). Traduzione e adattamento didattico per il percorso di orientamento all'informatica del Dipartimento di Informatica dell'Università degli Studi di Bari Aldo Moro.


In [ ]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download('https://github.com/AllenDowney/ThinkPython/raw/v3/thinkpython.py');
download('https://github.com/AllenDowney/ThinkPython/raw/v3/diagram.py');

import thinkpython

Nel 1939 Ernest Vincent Wright pubblicò *Gadsby*, un romanzo di 50.000 parole che non contiene la lettera «e». Poiché «e» è la lettera più frequente in inglese, scrivere anche poche parole senza utilizzarla è difficile. Per capire quanto, in questo capitolo calcoleremo quale frazione delle parole inglesi contiene almeno una «e».

A questo scopo useremo istruzioni `for` per scorrere le lettere di una stringa e le parole di un file, e aggiorneremo variabili all'interno di un ciclo per contare quante parole contengono una «e». Useremo l'operatore `in` per verificare se una lettera compare in una parola e imparerai uno schema di programmazione chiamato **ricerca lineare**.

Come esercizio, utilizzerai questi strumenti per risolvere un gioco di parole chiamato *Spelling Bee*.

## Cicli e stringhe

Nel Capitolo 3 abbiamo visto un ciclo `for` che usa la funzione `range` per visualizzare una sequenza di numeri.

In [ ]:
for i in range(3):
    print(i, end=' ')

Questa versione usa l’argomento con nome `end`, così la funzione `print` inserisce uno spazio dopo ogni numero anziché andare a capo.

Possiamo usare un ciclo `for` anche per visualizzare le lettere di una stringa.

In [ ]:
for letter in 'Gadsby':
    print(letter, end=' ')

Nota che ho cambiato il nome della variabile da `i` a `letter`, che fornisce più informazioni sul valore a cui si riferisce. La variabile definita in un ciclo `for` è chiamata **variabile di ciclo**.

Ora che sappiamo scorrere le lettere di una parola, possiamo verificare se contiene la lettera «e».

In [ ]:
for letter in "Gadsby":
    if letter == 'E' or letter == 'e':
        print('This word has an "e"')

Prima di proseguire, racchiudiamo questo ciclo in una funzione.

In [ ]:
def has_e():
    for letter in "Gadsby":
        if letter == 'E' or letter == 'e':
            print('This word has an "e"')

E trasformiamola in una funzione pura che restituisca `True` se la parola contiene una «e» e `False` altrimenti.

In [ ]:
def has_e():
    for letter in "Gadsby":
        if letter == 'E' or letter == 'e':
            return True
    return False

Possiamo generalizzarla facendo in modo che riceva la parola come parametro.

In [ ]:
def has_e(word):
    for letter in word:
        if letter == 'E' or letter == 'e':
            return True
    return False

Ora possiamo verificarne il funzionamento così:

In [ ]:
has_e('Gadsby')

In [ ]:
has_e('Emma')

## Lettura dell’elenco delle parole

Per sapere quante parole contengono una «e», ci serve un elenco di parole. Useremo un elenco di circa 114.000 parole ammesse nei cruciverba e in altri giochi di parole in inglese.

La cella seguente scarica l’elenco delle parole, una versione modificata di una raccolta realizzata da Grady Ward e resa di pubblico dominio nell’ambito del progetto Moby Lexicon (vedi [Moby Project](http://wikipedia.org/wiki/Moby_Project)).

In [ ]:
download('https://raw.githubusercontent.com/AllenDowney/ThinkPython/v3/words.txt');

L’elenco si trova in un file chiamato `words.txt`, scaricato nel notebook di questo capitolo. Per leggerlo useremo la funzione predefinita `open`, che riceve il nome del file e restituisce un **oggetto file** utilizzabile per leggerne il contenuto.

In [ ]:
file_object = open('words.txt')

L’oggetto file fornisce un metodo chiamato `readline`, che legge i caratteri fino al successivo carattere di fine riga e restituisce il risultato come stringa:

In [ ]:
file_object.readline()

Nota che la sintassi per chiamare `readline` è diversa da quella delle funzioni viste finora. Questo perché è un **metodo**, cioè una funzione associata a un oggetto. In questo caso `readline` è associato all’oggetto file: lo chiamiamo usando il nome dell’oggetto, l’operatore punto e il nome del metodo.

La prima parola dell’elenco è «aa», un tipo di lava. La sequenza `\n` rappresenta il carattere di fine riga che separa questa parola dalla successiva.

L’oggetto file tiene traccia della posizione raggiunta nel file: chiamando di nuovo `readline`, otteniamo la parola successiva:

In [ ]:
line = file_object.readline()
line

Per rimuovere il carattere di fine riga dalla parola possiamo usare `strip`, un metodo associato alle stringhe:

In [ ]:
word = line.strip()
word

`strip` rimuove i caratteri di spaziatura — inclusi spazi, tabulazioni e caratteri di fine riga — dall’inizio e dalla fine della stringa.

Possiamo anche usare un oggetto file in un ciclo `for`. Questo programma legge `words.txt` e visualizza ogni parola su una riga:

In [ ]:
for line in open('words.txt'):
    word = line.strip()
    print(word)

Ora che sappiamo leggere l’elenco delle parole, dobbiamo contarle. Per farlo, ci servirà la possibilità di aggiornare le variabili.

## Aggiornamento delle variabili

Come potresti aver scoperto, è possibile assegnare più di una volta un valore alla stessa variabile. Una nuova assegnazione fa sì che una variabile esistente si riferisca a un nuovo valore (e non più a quello precedente).

Ecco, per esempio, un’assegnazione iniziale che crea una variabile.

In [ ]:
x = 5
x

Ed ecco un’assegnazione che ne modifica il valore.

In [ ]:
x = 7
x

La figura seguente rappresenta queste assegnazioni in un diagramma di stato.

In [ ]:
from diagram import make_rebind, draw_bindings

bindings = make_rebind('x', [5, 7])

In [ ]:
from diagram import diagram, adjust

width, height, x, y = [0.54, 0.61, 0.07, 0.45]
ax = diagram(width, height)
bbox = draw_bindings(bindings, ax, x, y)
# adjust(x, y, bbox)

La freccia tratteggiata indica che `x` non si riferisce più a `5`; quella continua indica che ora si riferisce a `7`.

Un tipo comune di assegnazione è l’**aggiornamento**, in cui il nuovo valore della variabile dipende dal precedente.

In [ ]:
x = 7

In [ ]:
x = x + 1
x

Questa istruzione significa: «prendi il valore corrente di `x`, aggiungi uno e assegna il risultato nuovamente a `x`».

Se provi ad aggiornare una variabile inesistente, ottieni un errore, perché Python valuta l’espressione a destra prima di assegnare il valore alla variabile a sinistra.

In [ ]:
%%expect NameError

z = z + 1

Prima di aggiornare una variabile devi **inizializzarla**, di solito con una semplice assegnazione:

In [ ]:
z = 0
z = z + 1
z

Aumentare il valore di una variabile si chiama **incremento**; diminuirlo si chiama **decremento**. Poiché queste operazioni sono molto comuni, Python offre **operatori di assegnazione aumentata** che permettono di aggiornare una variabile in modo più conciso. Per esempio, `+=` incrementa una variabile della quantità indicata.

In [ ]:
z += 2
z

Esistono operatori di assegnazione aumentata anche per altre operazioni aritmetiche, tra cui `-=` e `*=`.

## Cicli e conteggi

Il programma seguente conta il numero di parole nell’elenco.

In [ ]:
total = 0

for line in open('words.txt'):
    word = line.strip()
    total += 1

Inizia inizializzando `total` a `0`. A ogni iterazione del ciclo incrementa `total` di `1`. Quando il ciclo termina, `total` contiene quindi il numero totale di parole.

In [ ]:
total

Una variabile di questo tipo, utilizzata per contare quante volte accade qualcosa, si chiama **contatore**.

Possiamo aggiungere un secondo contatore per tenere traccia delle parole che contengono una «e».

In [ ]:
total = 0
count = 0

for line in open('words.txt'):
    word = line.strip()
    total = total + 1
    if has_e(word):
        count += 1

Vediamo quante parole contengono una «e».

In [ ]:
count

Come percentuale di `total`, circa due terzi delle parole contengono la lettera «e».

In [ ]:
count / total * 100

Si capisce quindi perché sia difficile scrivere un intero libro senza utilizzare parole che contengono questa lettera.

## L’operatore `in`

La versione di `has_e` scritta in questo capitolo è più complicata del necessario. Python fornisce l’operatore `in`, che verifica se un carattere compare in una stringa.

In [ ]:
word = 'Gadsby'
'e' in word

Possiamo quindi riscrivere `has_e` così.

In [ ]:
def has_e(word):
    if 'E' in word or 'e' in word:
        return True
    else:
        return False

Poiché la condizione dell’istruzione `if` ha già un valore booleano, possiamo eliminare l’`if` e restituire direttamente quel valore.

In [ ]:
def has_e(word):
    return 'E' in word or 'e' in word

Possiamo semplificare ancora la funzione usando il metodo `lower`, che converte le lettere di una stringa in minuscolo. Ecco un esempio.

In [ ]:
word.lower()

`lower` crea una nuova stringa: non modifica quella esistente, quindi il valore di `word` rimane invariato.

In [ ]:
word

Ecco come usare `lower` in `has_e`.

In [ ]:
def has_e(word):
    return 'e' in word.lower()

In [ ]:
has_e('Gadsby')

In [ ]:
has_e('Emma')

## Ricerca

Partendo da questa versione più semplice di `has_e`, scriviamo una funzione più generale chiamata `uses_any`, che riceve un secondo parametro: una stringa di lettere. Restituisce `True` se la parola contiene almeno una delle lettere indicate e `False` altrimenti.

In [ ]:
def uses_any(word, letters):
    for letter in word.lower():
        if letter in letters.lower():
            return True
    return False

Ecco un esempio in cui il risultato è `True`.

In [ ]:
uses_any('banana', 'aeiou')

Ed eccone un altro in cui è `False`.

In [ ]:
uses_any('apple', 'xyz')

`uses_any` converte `word` e `letters` in minuscolo, quindi funziona con qualsiasi combinazione di maiuscole e minuscole.

In [ ]:
uses_any('Banana', 'AEIOU')

La struttura di `uses_any` è simile a quella di `has_e`: scorre le lettere di `word` e le controlla una alla volta. Se ne trova una presente in `letters`, restituisce immediatamente `True`. Se termina il ciclo senza trovarne, restituisce `False`.

Questo schema si chiama **ricerca lineare**. Negli esercizi alla fine del capitolo scriverai altre funzioni che lo utilizzano.

## Doctest

Nel [Capitolo 4](https://allendowney.github.io/ThinkPython/chap04.html) abbiamo usato una *docstring* per documentare una funzione, cioè per spiegarne il comportamento. Possiamo usare una docstring anche per **verificare** una funzione. Ecco una versione di `uses_any` con una docstring che contiene alcuni test.

In [ ]:
def uses_any(word, letters):
    """Checks if a word uses any of a list of letters.

    >>> uses_any('banana', 'aeiou')
    True
    >>> uses_any('apple', 'xyz')
    False
    """
    for letter in word.lower():
        if letter in letters.lower():
            return True
    return False

Ogni test inizia con `>>>`, il prompt utilizzato in alcuni ambienti Python per indicare dove inserire il codice. In un *doctest*, il prompt è seguito da un’espressione, di solito una chiamata di funzione. La riga successiva indica il valore atteso se la funzione funziona correttamente.

Nel primo esempio, `'banana'` contiene `'a'`, quindi il risultato dovrebbe essere `True`. Nel secondo, `'apple'` non contiene alcuna lettera di `'xyz'`, quindi il risultato dovrebbe essere `False`.

Per eseguire questi test dobbiamo importare il modulo `doctest` e chiamare una funzione chiamata `run_docstring_examples`. Per facilitarne l’uso, ho scritto la seguente funzione che riceve come argomento un oggetto funzione.

In [ ]:
from doctest import run_docstring_examples

def run_doctests(func):
    run_docstring_examples(func, globals(), name=func.__name__)

Non abbiamo ancora studiato `globals` e `__name__`: per ora puoi ignorarli. Ora possiamo verificare `uses_any` in questo modo.

In [ ]:
run_doctests(uses_any)

`run_doctests` trova le espressioni nella docstring e le valuta. Se il risultato coincide con quello atteso, il test **passa**; altrimenti **fallisce**.

Se tutti i test passano, `run_doctests` non mostra alcun output: in questo caso, nessuna notizia è una buona notizia. Per vedere che cosa succede quando un test fallisce, ecco una versione errata di `uses_any`.

In [ ]:
def uses_any_incorrect(word, letters):
    """Checks if a word uses any of a list of letters.

    >>> uses_any_incorrect('banana', 'aeiou')
    True
    >>> uses_any_incorrect('apple', 'xyz')
    False
    """
    for letter in word.lower():
        if letter in letters.lower():
            return True
        else:
            return False     # INCORRECT!

Ed ecco che cosa succede quando la sottoponiamo ai test.

In [ ]:
run_doctests(uses_any_incorrect)

L’output include l’esempio che non ha superato il test, il valore atteso e quello effettivamente prodotto dalla funzione.

Se non ti è chiaro perché il test sia fallito, avrai occasione di individuarne la causa in un esercizio.

## Glossario

**variabile di ciclo:** variabile definita nell’intestazione di un ciclo `for`.

**oggetto file:** oggetto che rappresenta un file aperto e tiene traccia delle parti già lette o scritte.

**metodo:** funzione associata a un oggetto e chiamata usando l’operatore punto.

**aggiornamento:** assegnazione che attribuisce un nuovo valore a una variabile già esistente anziché crearne una nuova.

**inizializzare:** creare una nuova variabile e assegnarle un valore.

**incremento:** aumento del valore di una variabile.

**decremento:** diminuzione del valore di una variabile.

**contatore:** variabile usata per contare qualcosa, normalmente inizializzata a zero e poi incrementata.

**ricerca lineare:** schema computazionale che esamina una sequenza di elementi e si ferma quando trova ciò che cerca.

**test superato:** test il cui risultato coincide con quello atteso (*pass*).

**test fallito:** test il cui risultato non coincide con quello atteso (*fail*).

## Esercizi

In [ ]:
# This cell tells Jupyter to provide detailed debugging information
# when a runtime error occurs. Run it before working on the exercises.

%xmode Verbose

### Chiedi a un assistente virtuale

In `uses_any` potresti aver notato che la prima istruzione `return` si trova dentro il ciclo e la seconda fuori.

In [ ]:
def uses_any(word, letters):
    for letter in word.lower():
        if letter in letters.lower():
            return True
    return False

Quando si scrivono per la prima volta funzioni di questo tipo, un errore comune consiste nel mettere entrambe le istruzioni `return` dentro il ciclo, come in questo esempio.

In [ ]:
def uses_any_incorrect(word, letters):
    for letter in word.lower():
        if letter in letters.lower():
            return True
        else:
            return False     # INCORRECT!

Chiedi a un assistente virtuale che cosa non funziona in questa versione.

### Esercizio

Scrivi una funzione chiamata `uses_none` che riceva una parola e una stringa di lettere proibite e restituisca `True` se la parola non contiene nessuna di quelle lettere.

Ecco uno scheletro della funzione con due doctest. Completa la funzione affinché superi i test e aggiungi almeno un altro doctest.

In [ ]:
def uses_none(word, forbidden):
    """Checks whether a word avoid forbidden letters.

    >>> uses_none('banana', 'xyz')
    True
    >>> uses_none('apple', 'efg')
    False
    """
    return None

In [ ]:
# Solution goes here

In [ ]:
run_doctests(uses_none)

### Esercizio

Scrivi una funzione chiamata `uses_only` che riceva una parola e una stringa di lettere e restituisca `True` se la parola contiene soltanto lettere presenti nella stringa.

Ecco uno scheletro della funzione con due doctest. Completa la funzione affinché superi i test e aggiungi almeno un altro doctest.

In [ ]:
def uses_only(word, available):
    """Checks whether a word uses only the available letters.

    >>> uses_only('banana', 'ban')
    True
    >>> uses_only('apple', 'apl')
    False
    """
    return None

In [ ]:
# Solution goes here

In [ ]:
run_doctests(uses_only)

### Esercizio

Scrivi una funzione chiamata `uses_all` che riceva una parola e una stringa di lettere e restituisca `True` se la parola contiene tutte le lettere indicate almeno una volta.

Ecco uno scheletro della funzione con due doctest. Completa la funzione affinché superi i test e aggiungi almeno un altro doctest.

In [ ]:
def uses_all(word, required):
    """Checks whether a word uses all required letters.

    >>> uses_all('banana', 'ban')
    True
    >>> uses_all('apple', 'api')
    False
    """
    return None

In [ ]:
# Solution goes here

In [ ]:
run_doctests(uses_all)

### Esercizio

Il *New York Times* pubblica ogni giorno un gioco chiamato *Spelling Bee*, in cui bisogna trovare quante più parole possibili utilizzando soltanto sette lettere, una delle quali è obbligatoria. Le parole devono avere almeno quattro lettere.

Per esempio, il giorno in cui è stato scritto questo capitolo, le lettere erano `ACDLORT`, con `R` obbligatoria. «color» è quindi una parola valida, mentre «told» non lo è perché non contiene `R`, e «rat» non lo è perché ha solo tre lettere. Le lettere possono ripetersi, quindi «ratatat» è valida.

Scrivi una funzione `check_word` che verifichi se una parola è ammissibile. Deve ricevere come parametri la parola, una stringa con le sette lettere disponibili e una stringa contenente la singola lettera obbligatoria. Puoi usare le funzioni scritte negli esercizi precedenti.

Ecco uno scheletro della funzione con alcuni doctest. Completa la funzione e verifica che tutti i test siano superati.

In [ ]:
def check_word(word, available, required):
    """Check whether a word is acceptable.

    >>> check_word('color', 'ACDLORT', 'R')
    True
    >>> check_word('ratatat', 'ACDLORT', 'R')
    True
    >>> check_word('rat', 'ACDLORT', 'R')
    False
    >>> check_word('told', 'ACDLORT', 'R')
    False
    >>> check_word('bee', 'ACDLORT', 'R')
    False
    """
    return False

In [ ]:
# Solution goes here

In [ ]:
run_doctests(check_word)

Secondo le regole di *Spelling Bee*:

- Le parole di quattro lettere valgono 1 punto ciascuna.
- Le parole più lunghe valgono 1 punto per lettera.
- Ogni partita contiene almeno un **pangramma**, cioè una parola che usa tutte le lettere disponibili: vale 7 punti aggiuntivi!

Scrivi una funzione `score_word` che riceva una parola e una stringa di lettere disponibili e ne restituisca il punteggio. Puoi supporre che la parola sia ammissibile.

Anche qui trovi uno scheletro della funzione con alcuni doctest.

In [ ]:
def word_score(word, available):
    """Compute the score for an acceptable word.

    >>> word_score('card', 'ACDLORT')
    1
    >>> word_score('color', 'ACDLORT')
    5
    >>> word_score('cartload', 'ACDLORT')
    15
    """
    return 0

In [ ]:
# Solution goes here

In [ ]:
run_doctests(word_score)

Quando tutte le funzioni superano i test, usa il ciclo seguente per cercare nell’elenco le parole ammissibili e sommarne i punteggi.

In [ ]:
available = 'ACDLORT'
required = 'R'

total = 0

file_object = open('words.txt')
for line in file_object:
    word = line.strip()
    if check_word(word, available, required):
        score = word_score(word, available)
        total = total + score
        print(word, score)

print("Total score", total)

Visita la pagina di [Spelling Bee](https://www.nytimes.com/puzzles/spelling-bee) e inserisci le lettere disponibili oggi. La lettera al centro è obbligatoria.

Ho trovato un insieme di lettere che permette di ottenere un punteggio totale di 5820. Riesci a fare meglio? Trovare l’insieme migliore potrebbe essere troppo difficile: occorre essere realisti.

### Esercizio

Potresti aver notato che le funzioni scritte negli esercizi precedenti hanno molto in comune. In effetti, sono così simili che spesso puoi usare una funzione per scriverne un’altra.

Per esempio, se una parola non contiene nessuna delle lettere proibite, significa che non ne contiene alcuna. Possiamo quindi scrivere una versione di `uses_none` in questo modo.

In [ ]:
def uses_none(word, forbidden):
    """Checks whether a word avoids forbidden letters.

    >>> uses_none('banana', 'xyz')
    True
    >>> uses_none('apple', 'efg')
    False
    >>> uses_none('', 'abc')
    True
    """
    return not uses_any(word, forbidden)

In [ ]:
run_doctests(uses_none)

Esiste anche una somiglianza tra `uses_only` e `uses_all` di cui puoi approfittare. Se hai una versione funzionante di `uses_only`, prova a scrivere una versione di `uses_all` che la chiami.

### Esercizio

Se ti sei bloccato nell’esercizio precedente, prova a chiedere a un assistente virtuale: «Data una funzione `uses_only` che riceve due stringhe e verifica che la prima usi soltanto lettere presenti nella seconda, usala per scrivere `uses_all`, che riceve due stringhe e verifica che la prima contenga tutte le lettere della seconda, consentendo ripetizioni».

Usa `run_doctests` per verificare la risposta.

In [ ]:
# Solution goes here

In [ ]:
run_doctests(uses_all)

### Esercizio

Vediamo ora se possiamo scrivere `uses_all` partendo da `uses_any`.

Chiedi a un assistente virtuale: «Data una funzione `uses_any` che riceve due stringhe e verifica se la prima contiene almeno una lettera della seconda, puoi usarla per scrivere `uses_all`, che riceve due stringhe e verifica se la prima contiene tutte le lettere della seconda, consentendo ripetizioni?».

Se risponde di sì, assicurati di verificare il risultato!

In [ ]:
# Solution goes here

In [ ]:
# Here's what I got from ChatGPT 4o December 26, 2024
# It's correct, but it makes multiple calls to uses_any

def uses_all(s1, s2):
    """Checks if all characters in s2 are in s1, allowing repeats."""
    for char in s2:
        if not uses_any(s1, char):
            return False
    return True

In [ ]:
# Solution goes here